In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------
import requests
import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import os

from selenium.webdriver.common.keys import Keys

from zipfile import ZipFile

import urllib3
from selenium.webdriver.support.ui import Select
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ZA FSCA' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

# writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {

            'ZA FSCA 1': 'https://www2.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=List_Of_Registered_Insurers',

            'ZA FSCA 2': 'https://www2.fsca.co.za/MagicScripts/mgrqispi.dll',

            'ZA FSCA 3': 'https://www2.fsca.co.za/MagicScripts/mgrqispi.dll',

            'ZA FSCA 4': 'https://prod-entitysearchwebapplication.azurewebsites.net/legacy-searches',

            'ZA FSCA 5': 'https://www2.fsca.co.za/MagicScripts/mgrqispi.dll',

            'ZA FSCA 6': 'https://www2.fsca.co.za/MagicScripts/mgrqispi.dll'

           }


Typology={

       regulatorName + ' 1': 'List of Licenced Insurers',
       regulatorName + ' 2': 'List of Local Collective Investment Schemes',
       regulatorName + ' 3': 'List of Foreign Collective Investment Schemes',
       regulatorName + ' 4': 'List of Registered Credit Rating Agencies',
       regulatorName + ' 5': 'List of Registered Active Funds',
       regulatorName + ' 6': 'List of Approved Administrators ',

        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}

columns = ['Payment institution', 'Competent', 'Method', 'Payment', 'Date']

processdate = now.strftime('%Y-%m-%d')




Running ZA FSCA Web Scraping Tool v.1.1


In [2]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def unzip(source_path, output_path):

    with ZipFile(source_path, 'r') as zip_:

        zip_.extractall(output_path) 




In [ ]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg}")

    driver.get(regdict[reg])

    sleep(1)

    soup=BeautifulSoup(driver.page_source, 'html.parser')

    sleep(0.2)


    if reg=='ZA FSCA 1'  :

        for br in soup.find_all("br"):

            br.replace_with("|")

        tbody = soup.find_all('tbody')[-1].find_all('tr')

        print(f"[INFO] : - len Table = {len(tbody[1:])}")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')

            sqldict['Name'].append(td[1].text.split('|')[0])

            sqldict['Address_1'].append(td[1].text.split('|')[1])

            sqldict['Phone'].append(td[2].text)

            sqldict['InternalID_1'].append(td[0].text)

            sqldict['InternalID_1_type'].append('INSURER NO')

            sqldict['InternalID_2'].append(td[4].text)

            sqldict['InternalID_2_type'].append('INSURER REGISTERED NUMBER')

            sqldict['ListName'].append(Typology[reg])

            sqldict['Typology'].append(td[3].text.strip())

            sqldict['Cntry'].append('ZA')

            sqldict['RegulationType'].append('Regulated')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict)   

    elif reg == 'ZA FSCA 2' or reg == 'ZA FSCA 3' :

        opt = 'L' if reg == 'ZA FSCA 2' else 'F'


        data = {
            "Manco_Name": "",
            "Scheme_Name": "",
            "Company_Type": "",
            "Local_Foreign": opt,  # example from your payload
            "APPNAME": "Web",
            "PRGNAME": "Display_Results",
            "ARGUMENTS": "Manco_Name,Scheme_Name,Portfolio_Name",
            "bSubmit": "Please Wait!"
        }

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
            "Content-Type": "application/x-www-form-urlencoded",
            "Origin": "https://www2.fsca.co.za",
            "Referer": "https://www2.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=Search_Mancos"
        }

        resp = requests.post(regdict[reg], data=data, headers=headers, timeout=30)
        resp.raise_for_status()
        sleep(1)

        soup=BeautifulSoup(resp.text, 'html.parser')

        tbody = soup.find_all('table')[-1].find_all('tr')

        print(f"[INFO] : - Table containe = {len(tbody[1:])} rows")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')
            Num_ = td[0].text
            # manager_name = td[1].text
            scheme_name = td[-5].text
            company_type = td[-4].text
            detail_url = "https://www2.fsca.co.za/MagicScripts/mgrqispi.dll"
            data = {
                "Manco_No": str(Num_),
                "APPNAME": "Web",
                "PRGNAME": "Display_Manco_Details",
                "ARGUMENTS": "Manco_No"
            }

            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,"
                        "image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
                "Content-Type": "application/x-www-form-urlencoded",
                "Origin": "https://www2.fsca.co.za",
                "Referer": "https://www2.fsca.co.za/MagicScripts/mgrqispi.dll"
            }
            resp_detail = requests.post(detail_url, data=data, headers=headers, timeout=30, verify=False)
            soup2 = BeautifulSoup(resp_detail.text, 'html.parser')
            for br in soup2.find_all("br"):
                br.replace_with(" ")
            tables = soup2.find_all('table')
            manager_name = tables[0].find_all('tr')[1].find('td').text if tables[0].find_all('tr')[1].find('th').text == 'Manager Name' else ''
            Company_No = tables[0].find_all('tr')[-2].find('td').text if tables[0].find_all('tr')[-2].find('th').text == 'Company No' else ''
            Address = tables[1].find_all('tr')[0].find('td').text
            Phone = tables[1].find_all('tr')[1].find('td').text
            #print(f"Manager Name: {manager_name}, Scheme Name: {scheme_name}, Company Type: {company_type}, "f"Company No: {Company_No}, Address: {Address}, Phone: {Phone}")
            sqldict['Name - Mother Company'].append(manager_name)
            sqldict['Name'].append(scheme_name)
            sqldict['InternalID_1'].append(Company_No)
            sqldict['InternalID_1_type'].append('Company Number') if len(Company_No)>0 else sqldict['InternalID_1_type'].append('')
            sqldict['Address_1'].append(Address)
            sqldict['Phone'].append(Phone)
            sqldict['Typology'].append(company_type)
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict = bourange_same_length_array(sqldict)

    elif reg == 'ZA FSCA 4':

        sleep(30)
        drop=Select(driver.find_element(By.CLASS_NAME, 'nav-select'))
        drop.select_by_value("1")
        sleep(1)
        soup=BeautifulSoup(driver.page_source, 'html.parser')

        tbody = soup.find('tbody').find_all('tr')

        print(f"[INFO] : - len Table = {len(tbody[1:])}")

        for i, tr in enumerate(tbody[:]):

            td = tr.find_all('td')



            if td[2].text.find('Registered') != -1 :
                sqldict['Name'].append(td[0].text)
                sqldict['Cntry'].append(td[1].text)

                sqldict['RegulationType'].append(td[2].text)

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0]) 

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict) 

    elif reg == 'ZA FSCA 5':
        start_list = ['#','A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S','T','U','V','W','X','Y','Z']
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
            "Content-Type": "application/x-www-form-urlencoded",
            "Origin": "https://www2.fsca.co.za",
            "Referer": "https://www2.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=Search_Mancos"
        }

        for b_0 in start_list:
            print(f"[INFO] : - Starting with {b_0}")
            data = {
                "Starting_With": b_0,
                "Fund_No": "",
                "APPNAME": "Web",
                "PRGNAME": 'List_Approved_Retire_Funds',  # example from your payload
                "ARGUMENTS": "Starting_With",
                "b0": b_0,
            }


            resp = requests.post(regdict[reg], data=data, headers=headers, timeout=30)
            resp.raise_for_status()
            sleep(1)

            soup=BeautifulSoup(resp.text, 'html.parser')

            tbody = soup.find_all('table')[-1].find_all('tr')

            print(f"[INFO] : - Table containe = {len(tbody[1:])} rows")

            for i, tr in enumerate(tbody[1:]):
                td = tr.find_all('td')
                # print(tr)
                if 'NORMAL ACTIVE FUND' in td[4].text:
                    #print(td[0].text, td[1].text, td[2].text, td[3].text, td[4].text)
                    fund_nr = td[0].text
                    fund_name = td[1].text
                    registered_address = td[2].text
                    type_ = td[3].text
                    mother_company = td[5].text
                    sqldict['Name'].append(fund_name)
                    sqldict['Address_1'].append(registered_address)
                    sqldict['Typology'].append(type_)
                    sqldict['Name - Mother Company'].append(mother_company)
                    sqldict['InternalID_1'].append(fund_nr) 
                    sqldict['InternalID_1_type'].append('Fund Number')
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split(' ')[0]) 
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict = bourange_same_length_array(sqldict) 
    elif reg == 'ZA FSCA 6':
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
            "Content-Type": "application/x-www-form-urlencoded",
            "Origin": "https://www2.fsca.co.za",
            "Referer": "https://www2.fsca.co.za/MagicScripts/mgrqispi.dll?APPNAME=Web&PRGNAME=Search_Mancos"
        }


        data = {
            "Function_No": "-1",
            "Admin_No": "",
            "Admin_Name": "",
            "APPNAME": "Web",
            "Empty_String": "",
            "User_No": "",
            "PRGNAME": 'Interm_list', 
            "ARGUMENTS": "Function_No,Admin_No,Admin_Name",
            "bSubmit":'Submit'
        }


        resp = requests.post(regdict[reg], data=data, headers=headers, timeout=30)
        resp.raise_for_status()
        sleep(1)

        soup=BeautifulSoup(resp.text, 'html.parser')

        for br in soup.find_all("br"):

            br.replace_with("|")

        tbody = soup.find_all('tbody')[-1].find_all('tr')

        print(f"[INFO] : - len Table = {len(tbody[1:])}")

        for i, tr in enumerate(tbody[1:]):

            td = tr.find_all('td')

            sqldict['Name'].append(td[0].text.split('|')[0])

            sqldict['Address_1'].append(td[0].text.split('|')[1])

            sqldict['Phone'].append(td[2].text)

            sqldict['InternalID_1'].append(td[1].text)

            sqldict['InternalID_1_type'].append('REGISTRATION NO')

            sqldict['ListName'].append(Typology[reg])

            sqldict['Typology'].append(td[-1].text.strip())

            sqldict['RegulationType'].append('Regulated')

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

        sqldict = bourange_same_length_array(sqldict)   






[INFO] : Working 1/1 | ZA FSCA 6
[INFO] : - len Table = 110


In [ ]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(filename, index=False)

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


    